Constuction du dataframe à partir du dossier contenants tous les DXF

In [8]:
import logging
import ezdxf
import pandas as pd
from pathlib import Path

def _safe_utf8(s):
    """Enlève les caractères surrogate / invalides UTF-8 qui provoquent UnicodeEncodeError."""
    if not isinstance(s, str):
        return s
    return s.encode("utf-8", errors="replace").decode("utf-8")

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger(__name__)

dxfs_dir = Path(r"C:\Users\mvm\Geolux_CV_Clone\dxf_test")
if not dxfs_dir.is_dir():
    raise FileNotFoundError(f"Dossier introuvable : {dxfs_dir}")

dxf_files = sorted(dxfs_dir.glob("*.dxf"), key=lambda p: p.name.lower())
total_files = len(dxf_files)
logger.info("Début lecture DXF : %d fichier(s) dans %s", total_files, dxfs_dir)

all_rows = []
failed_files = []

for i, path in enumerate(dxf_files, start=1):
    name = path.name
    try:
        doc = ezdxf.readfile(str(path))
        msp = doc.modelspace()
        count = 0
        for e in msp:
            row = {
                "file_name": _safe_utf8(name),
                "entity_type": _safe_utf8(e.dxftype()),
                "handle": getattr(e.dxf, "handle", None),
            }
            for key, value in e.dxfattribs().items():
                if value is None or isinstance(value, (int, float, bool)):
                    row[key] = value
                elif isinstance(value, str):
                    row[key] = _safe_utf8(value)
                else:
                    row[key] = _safe_utf8(str(value))
            all_rows.append(row)
            count += 1
        logger.info("[%d/%d] %s — OK (%d entités)", i, total_files, _safe_utf8(name), count)
    except Exception as ex:
        failed_files.append((_safe_utf8(name), _safe_utf8(str(ex))))
        logger.error("[%d/%d] %s — Échec : %s", i, total_files, _safe_utf8(name), _safe_utf8(str(ex)))

df_origin = pd.DataFrame(all_rows)
logger.info(
    "Terminé : %d fichier(s) lu(s), %d entité(s), %d échec(s)%s",
    total_files - len(failed_files),
    len(all_rows),
    len(failed_files),
    f" — {[f[0] for f in failed_files]}" if failed_files else "",
)

16:39:21 [INFO] Début lecture DXF : 2001 fichier(s) dans C:\Users\mvm\Geolux_CV_Clone\dxf_test
16:39:21 [INFO] creating ACAD_COLOR dictionary
16:39:21 [INFO] [1/2001] -1. KG.dxf — OK (2405 entités)
16:39:22 [INFO] creating ACAD_COLOR dictionary
16:39:22 [INFO] [2/2001] -4. Lageplan-1.dxf — OK (541 entités)
16:39:22 [INFO] [3/2001] 0. EG.dxf — OK (3519 entités)
16:39:24 [INFO] [4/2001] 001_ SS_RDC.dxf — OK (4606 entités)
16:39:26 [INFO] [5/2001] 002_ E1_E2.dxf — OK (7327 entités)
16:39:26 [INFO] creating ACAD_COLOR dictionary
16:39:26 [INFO] [6/2001] 003_ E3_TOI.dxf — OK (3386 entités)
16:39:35 [INFO] creating ACAD_COLOR dictionary
16:39:36 [INFO] [7/2001] 004_ FACPRI_FACARR_FACLAT.dxf — OK (81723 entités)
16:39:40 [INFO] creating ACAD_COLOR dictionary
16:39:40 [INFO] [8/2001] 005_ COU AA_COU BB_PDM.dxf — OK (23706 entités)
16:39:41 [INFO] creating ACAD_COLOR dictionary
16:39:42 [INFO] [9/2001] 01 Plans rez + sous-sol.dxf — OK (12093 entités)
16:39:42 [INFO] creating ACAD_COLOR dictiona

Sauvegarde du dataframe en un CSV (+- 3Go)

In [12]:
# Sauvegarde de df_origin en CSV dans le répertoire courant
from pathlib import Path
out_path = Path.cwd() / "geolux_client_raw.csv"
df_origin.to_csv(out_path, index=False, encoding="utf-8")
print(f"Sauvegardé : {out_path} ({len(df_origin):,} lignes)")

Sauvegardé : c:\Users\mvm\open3d_vision\src\geolux_client_raw.csv (8,269,883 lignes)


Exploration des données

In [38]:
for col in df_origin.columns:
    non_null_vals = df_origin[col][df_origin[col].notna()].head(5)
    if not non_null_vals.empty:
        print(f"{col} :")
        print(non_null_vals.values)
        print("-" * 40)

file_name :
<ArrowStringArray>
['-1. KG.dxf', '-1. KG.dxf', '-1. KG.dxf', '-1. KG.dxf', '-1. KG.dxf']
Length: 5, dtype: str
----------------------------------------
entity_type :
<ArrowStringArray>
['INSERT', 'INSERT', 'INSERT', 'INSERT', 'INSERT']
Length: 5, dtype: str
----------------------------------------
handle :
<ArrowStringArray>
['76', '80', '8A', '94', '9E']
Length: 5, dtype: str
----------------------------------------
owner :
<ArrowStringArray>
['1F', '1F', '1F', '1F', '1F']
Length: 5, dtype: str
----------------------------------------
layer :
<ArrowStringArray>
['räume', 'räume', 'räume', 'räume', 'räume']
Length: 5, dtype: str
----------------------------------------
linetype :
<ArrowStringArray>
['Continuous', 'Continuous', 'Continuous', 'Continuous', 'Continuous']
Length: 5, dtype: str
----------------------------------------
lineweight :
[13. 13. 13. 13. 13.]
----------------------------------------
color :
[7. 7. 7. 7. 7.]
----------------------------------------
att

nettoyage des colonnes inexploitables (une seule valeur au total)

In [43]:
# Supprimer les colonnes avec une seule valeur différente (hors NaN)
colonnes_a_supprimer = []
for col in df_origin.columns:
    # Compte les valeurs uniques hors NaN
    uniques = df_origin[col].dropna().unique()
    nb_uniques = len(uniques)
    if nb_uniques == 1:
        colonnes_a_supprimer.append(col)
    elif nb_uniques == 2:
        print(f"Colonne avec 2 valeurs différentes : {col}")
        print(uniques)
        print("-" * 40)

# Suppression effective des colonnes non-informatives
df_origin.drop(columns=colonnes_a_supprimer, inplace=True)
print(f"{len(colonnes_a_supprimer)} colonnes supprimées (une seule valeur différente) :")
print(colonnes_a_supprimer)

Colonne avec 2 valeurs différentes : solid_fill
[0. 1.]
----------------------------------------
Colonne avec 2 valeurs différentes : associative
[1. 0.]
----------------------------------------
Colonne avec 2 valeurs différentes : pattern_double
[0. 1.]
----------------------------------------
Colonne avec 2 valeurs différentes : flow_direction
[1. 5.]
----------------------------------------
Colonne avec 2 valeurs différentes : line_spacing_style
[1. 2.]
----------------------------------------
Colonne avec 2 valeurs différentes : clipping
[1. 0.]
----------------------------------------
Colonne avec 2 valeurs différentes : brightness
[50. 59.]
----------------------------------------
Colonne avec 2 valeurs différentes : clipping_boundary_type
[2. 1.]
----------------------------------------
Colonne avec 2 valeurs différentes : version
[0. 2.]
----------------------------------------
Colonne avec 2 valeurs différentes : degree
[3. 2.]
----------------------------------------
Colonne 

In [ ]:
layers_geo = ["TERRASSES",
              "MUR PORTEUR", 
              "CLOISONS",
              "PORTES",
              "FENETRES",
              "A SUPPRIMER",
              "ESCALIERS",
              "COUPE",
              "TEXTE-LOT",
              "SURFACE-LOT",
              "LIMITE PARCELLAIRE",
              "TEXTE-LOT-NUMERO",
              "EXTERIEUR",
              "CADRE-CARTOUCHE",
              "PARKING",
              "HACHURES",
              "TEXTE",
              "HAUTEUR 1-2m",
              "Par défaut",
              "COTATIONS MUR",
              "COTATIONS"]
layers_to_keep = ["TERRASSES", "MUR PORTEUR", "CLOISONS", "PORTES", "FENETRES", "ESCALIERS", "COTATION"]
hachures_alternatives = [ "HACHURE", "Schraffur","hatching"]
cotations_alternatives = ["COTATION","quoting","zitieren"]
cotation_murs_alternatives = ["COTATION MUR"]
terrasses_alternatives = [
    
    "TERRASSE", "AUSSENBEREICH", "TERRASSENBEREICH",  # allemand
    "TERRASE","TERRACE"
]
murs_porteurs_alternatives = [
    "MUR", 
    "TRAG", "MAUER", "WAND",  # allemand
    "PORTANTE", "Räume","WALL"
]
cloisons_alternatives = [
    "CLOISON",
    "TRENNWAN", "INNENWAN",  # allemand
    "CLOISON", "PAROI"
]
portes_alternatives = [
    "PORTE", "PORTAIL",
    "TÜR","DOOR"
]
fenetres_alternatives = [
    "FENETRE", "OUVERTURE",
    "FENSTER","BAY","WINDOW"
]
escaliers_alternatives = [
    "ESCALIER", "STAIRS"
    "TREPPE", "HAUPTTREPPE", "STIEGE",  # allemand
    
]
coupes_alternatives = ["COUPE","CUT","schneiden"]
textes_alternatives = ["TEXT","TEXTEN"]
surfaces_alternatives = ["SURFACE","AREA"]
limites_parcellaires_alternatives = ["LIMITE","GRENZE","PARCELLE"]
parkings_alternatives = ["PARKING","PARKHAUS"]

alternatives = [terrasses_alternatives, 
                murs_porteurs_alternatives, 
                cloisons_alternatives, 
                portes_alternatives, 
                fenetres_alternatives, 
                escaliers_alternatives,
                hachures_alternatives,
                cotations_alternatives,
                cotation_murs_alternatives,
                coupes_alternatives,
                textes_alternatives,
                surfaces_alternatives,
                limites_parcellaires_alternatives,
                parkings_alternatives
                ]

                

def lower_array(array):
    return [str(value).lower() for value in array]

layers_geo = lower_array(layers_geo)



Associe les numéros 1 à 6 aux listes d’alternatives, dans l’ordre du notebook :

1 = terrasses_alternatives

2 = murs_porteurs_alternatives

3 = cloisons_alternatives

4 = portes_alternatives

5 = fenetres_alternatives

6 = escaliers_alternatives

7 = hachures_alternatives

8 = cotations_alternatives

9 = cotations_murs_alternatives

10 = coupes_alternatives

11 = textes_alternatives

12 = surfaces_alternatives

13 = limites_parcellaires_alternatives

14 = parking_alternatives


In [149]:
# Colonne target : 0 par défaut, 1..6 selon la liste d'alternatives correspondante
# Ordre : 1=terrasses, 2=murs_porteurs, 3=cloisons, 4=portes, 5=fenetres, 6=escaliers
# On initialise la colonne "target" à 0 pour toutes les lignes
df_origin["target"] = 0

# On parcourt chaque liste d'alternatives avec son identifiant cible (target_id de 1 à 6 ici)
for target_id, alt_list in enumerate(alternatives, start=1):
    # Convertit chaque mot de la liste d'alternatives en minuscule et enlève les espaces inutiles
    allowed = [str(s).lower().strip() for s in alt_list]
    # On crée un masque pour sélectionner les lignes où "target" vaut encore 0
    mask_target_0 = (df_origin["target"] == 0)
    # Pour chaque mot-clé, on cherche s'il est *contenu* (et non équivalent) dans 'layer'
    for motif in allowed:
        # .str.contains vérifie si le motif apparait dans le texte du "layer" (case insensitive)
        mask_contains = df_origin["layer"].str.contains(motif, na=False)
        mask = mask_target_0 & mask_contains
        df_origin.loc[mask, "target"] = target_id
# Récap

# Calcule et affiche le pourcentage de target == 0 par rapport au total
target_counts = df_origin["target"].value_counts().sort_index()
percent_target_0 = (target_counts.get(0, 0) / target_counts.sum()) * 100
print(f"Pourcentage de target=0 : {percent_target_0:.2f}% (par rapport à toutes les valeurs)")
print("Détail des effectifs par valeur target :\n", target_counts)

Pourcentage de target=0 : 76.94% (par rapport à toutes les valeurs)
Détail des effectifs par valeur target :
 target
0     6362580
1        4645
2      808303
3      172062
4       81605
5       93462
6       57475
7       53674
8      196104
10     107758
11     113548
12      56307
13     137833
14      24527
Name: count, dtype: int64


In [139]:
# Modalités avec le plus d'occurrences dans df_origin["layer"], uniquement pour les lignes où "target" vaut 0
layer_counts = df_origin.loc[df_origin["target"] == 0, "layer"].value_counts()
print(f"Top 50 calques par nombre d'occurrences (parmi {len(layer_counts)} modalités au total, pour target==0) :")
layer_counts.head(60)

Top 50 calques par nombre d'occurrences (parmi 12334 modalités au total, pour target==0) :


layer
points                                                                                       363747
motif                                                                                        341434
0                                                                                            316238
sf_moti                                                          motif                       315676
facade                                                                                       314503
hachurage 1                                                                                  292206
de_a_veg                                                         aménag. ext._ végétation    249581
neubau_50 möblierung _ mobilier.a+a                                                          229701
voiles béton                                                                                 179081
standard                                                                                     1

In [159]:
# Applique lower_array à chaque colonne de type chaîne de caractère dans df_origin
for col in df_origin.select_dtypes(include="str").columns:
    df_origin[col] = lower_array(df_origin[col])


C:\Users\mvm\AppData\Local\Temp\ipykernel_29884\1347830748.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_origin.select_dtypes(include="object").columns:


In [160]:
# Tableau associatif : layer (original) -> nom du target
# Noms des targets dans le même ordre que les listes d'alternatives (1 à 14)
TARGET_NAMES = [
    "TERRASSES", "MUR PORTEUR", "CLOISONS", "PORTES", "FENETRES", "ESCALIERS",
    "HACHURES", "COTATION", "COTATION MUR", "COUPE", "TEXTE", "SURFACE",
    "LIMITE PARCELLAIRE", "PARKING"
]
target_id_to_name = {0: "autre"}
target_id_to_name.update({i: TARGET_NAMES[i - 1] for i in range(1, len(TARGET_NAMES) + 1)})

# Une ligne par (layer, target) unique, avec le nom du target
tableau_layer_target = (
    df_origin[:]
    .drop_duplicates()
    .assign(target_nom=lambda df: df["target"].map(target_id_to_name))
    .sort_values(["target", "layer"])
    .reset_index(drop=True)
)
tableau_layer_target

,file_name,entity_type,handle,owner,layer,linetype,lineweight,color,name,insert,...,default_end_width,thickness,fit_tolerance,bg_fill_color_name,paperspace,start_tangent,end_tangent,unit_vector,target,target_nom
0,034_diek_aut_fac_2024.06.06.dxf,lwpolyline,e5c7,1f,,0,NaN,114.0,nan,nan,...,NaN,NaN,NaN,nan,NaN,nan,nan,nan,0,autre
1,034_diek_aut_fac_2024.06.06.dxf,lwpolyline,e654,1f,,0,NaN,114.0,nan,nan,...,NaN,NaN,NaN,nan,NaN,nan,nan,nan,0,autre
2,034_diek_aut_fac_2024.06.06.dxf,mtext,e690,1f,,,NaN,114.0,nan,"(453.1794274376368, 26.83150764608996, 0.0)",...,NaN,NaN,NaN,nan,NaN,nan,nan,nan,0,autre
3,034_diek_aut_fac_2024.06.06.dxf,lwpolyline,e6cb,1f,,0,NaN,114.0,nan,nan,...,NaN,NaN,NaN,nan,NaN,nan,nan,nan,0,autre
4,034_diek_aut_fac_2024.06.06.dxf,mtext,e6ce,1f,,,NaN,114.0,nan,"(453.1794274376368, 26.83150764608996, 0.0)",...,NaN,NaN,NaN,nan,NaN,nan,nan,nan,0,autre
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8269878,rew1t0709cad.dxf,lwpolyline,edc7,1f,voirie parking velo,nan,0.0,NaN,nan,nan,...,NaN,NaN,NaN,nan,NaN,nan,nan,nan,14,PARKING
8269879,siu2k0602sie+cn.dxf,lwpolyline,17d90,1f,voirie parking velo,nan,0.0,NaN,nan,nan,...,NaN,NaN,NaN,nan,NaN,nan,nan,nan,14,PARKING
8269880,siu_limites.dxf,lwpolyline,edc7,1f,voirie parking velo,nan,0.0,NaN,nan,nan,...,NaN,NaN,NaN,nan,NaN,nan,nan,nan,14,PARKING
8269881,uah2x1002sie.dxf,lwpolyline,edc7,1f,voirie parking velo,nan,0.0,NaN,nan,nan,...,NaN,NaN,NaN,nan,NaN,nan,nan,nan,14,PARKING


In [161]:
# Sauvegarde de df_origin en CSV dans le répertoire courant

out_path = Path.cwd() / "layer_raw_and_target_full.csv"
tableau_layer_target.to_csv(out_path, index=False, encoding="utf-8")
print(f"Sauvegardé : {out_path} ({len(df_origin):,} lignes)")

Sauvegardé : c:\Users\mvm\open3d_vision\src\layer_raw_and_target_full.csv (8,269,883 lignes)


In [162]:
i = 0
j = 0
word_used = []
word_not_used = []
# Boucle sur chaque sous-liste d'alternatives (chaque catégorie)
for row in alternatives:
    print(row)  # Affiche la sous-liste d'alternatives en cours
    print(type(row))  # Affiche le type de la sous-liste (devrait être list)
    # Parcourt chaque mot-clef de la sous-liste
    for word in row:
        word = word.lower()  # Met le mot en minuscules pour harmoniser la recherche
        print(word)  # Affiche le mot-clef en cours
        # Vérifie si le mot-clef existe dans n'importe quelle valeur de la colonne "layer" du DataFrame
        if df_origin["layer"].str.contains(word, na=False).any():
            i += 1  # Incrémente le compteur de mots retrouvés dans les calques
            word_used.append(word)  # Ajoute le mot à la liste des mots retrouvés
        else:
            j += 1  # Incrémente le compteur de mots non retrouvés dans les calques
            word_not_used.append(word)  # Ajoute le mot à la liste des mots non retrouvés
print(i)  # Affiche le nombre total de mots retrouvés
print(j)

['TERRASSE', 'AUSSENBEREICH', 'TERRASSENBEREICH', 'TERRASE', 'TERRACE']
<class 'list'>
terrasse
aussenbereich
terrassenbereich
terrase
terrace
['MUR', 'TRAG', 'MAUER', 'WAND', 'PORTANTE', 'Räume', 'WALL']
<class 'list'>
mur
trag
mauer
wand
portante
räume
wall
['CLOISON', 'TRENNWAN', 'INNENWAN', 'CLOISON', 'PAROI']
<class 'list'>
cloison
trennwan
innenwan
cloison
paroi
['PORTE', 'PORTAIL', 'TÜR', 'DOOR']
<class 'list'>
porte
portail
tür
door
['FENETRE', 'OUVERTURE', 'FENSTER', 'BAY', 'WINDOW']
<class 'list'>
fenetre
ouverture
fenster
bay
window
['ESCALIER', 'STAIRSTREPPE', 'HAUPTTREPPE', 'STIEGE']
<class 'list'>
escalier
stairstreppe
haupttreppe
stiege
['HACHURE', 'Schraffur', 'hatching']
<class 'list'>
hachure
schraffur
hatching
['COTATION', 'quoting', 'zitieren']
<class 'list'>
cotation
quoting
zitieren
['COTATION MUR']
<class 'list'>
cotation mur
['COUPE', 'CUT', 'schneiden']
<class 'list'>
coupe
cut
schneiden
['TEXT', 'TEXTEN']
<class 'list'>
text
texten
['SURFACE', 'AREA']
<class '